# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv()) 

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [7]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### Example 1

#### Create our QandA application

In [9]:
pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.6 MB/s eta 0:00:00


In [11]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.4 MB/s eta 0:00:00


In [14]:
pip install langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [15]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain


In [17]:
file = 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [ ]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [19]:
pip install docarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 26.7 MB/s eta 0:00:00


In [22]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cuda'})
).from_loaders([loader])

In [23]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [24]:
data[10]

Document(metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 10}, page_content='\ufeffid: AVqVGZNvQMlgsOJE6eUY\ndateAdded: 2017-03-03T16:56:05Z\ndateUpdated: 2018-10-25T16:36:31Z\nname: Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)\nasins: B00ZV9PXP2\nbrand: Amazon\ncategories: Computers,Electronics Features,Tablets,Electronics,iPad & Tablets,Kindle E-readers,iPad Accessories,Used:Tablets,E-Readers,E-Readers & Accessories,Computers/Tablets & Networking,Used:Computers Accessories,iPads Tablets,All Tablets,Tablets & E-readers,Computers & Tablets,Amazon,Tablets & eBook Readers\nprimaryCategories: Electronics\nimageURLs: https://pisces.bbystatic.com/image2/BestBuy_US/images/products/5442/5442403_sd.jpg,https://c1.neweggimages.com/NeweggImage/ProductImage/A3FA_1_201801081360871160.jpg,https://i.ebayimg.com/thumbs/images/g/N4IAAOSwoA9Zgkso/s-l96.jpg,http://i.ebayimg.com/thumbs/images/g/dpkAAOSwfpVZFKHy/s-l200.jpg,http://i.ebayimg.com/thumbs/images/

In [25]:
data[11]

Document(metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 11}, page_content='\ufeffid: AVqVGZNvQMlgsOJE6eUY\ndateAdded: 2017-03-03T16:56:05Z\ndateUpdated: 2018-10-25T16:36:31Z\nname: Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)\nasins: B00ZV9PXP2\nbrand: Amazon\ncategories: Computers,Electronics Features,Tablets,Electronics,iPad & Tablets,Kindle E-readers,iPad Accessories,Used:Tablets,E-Readers,E-Readers & Accessories,Computers/Tablets & Networking,Used:Computers Accessories,iPads Tablets,All Tablets,Tablets & E-readers,Computers & Tablets,Amazon,Tablets & eBook Readers\nprimaryCategories: Electronics\nimageURLs: https://pisces.bbystatic.com/image2/BestBuy_US/images/products/5442/5442403_sd.jpg,https://c1.neweggimages.com/NeweggImage/ProductImage/A3FA_1_201801081360871160.jpg,https://i.ebayimg.com/thumbs/images/g/N4IAAOSwoA9Zgkso/s-l96.jpg,http://i.ebayimg.com/thumbs/images/g/dpkAAOSwfpVZFKHy/s-l200.jpg,http://i.ebayimg.com/thumbs/images/

#### Hard-coded examples

In [26]:
from langchain.prompts import PromptTemplate

In [27]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do customers like the battery life of this product?",
        "answer": "Yes"
    },
    {
        "query": "Are users satisfied with the price?",
        "answer": "No"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do customers like the battery life of this product?\n"
             "   Answer: Yes\n"
             "2. Query: Are users satisfied with the price?\n"
             "   Answer: No\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the product quality generally considered good?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


<ipython-input-27-39ab3ed7a37b>:44: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(
<ipython-input-27-39ab3ed7a37b>:54: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm_chain.run({"query": query})


answer='Yes'


#### LLM-Generated examples

In [28]:
from langchain.evaluation.qa import QAGenerateChain

In [29]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [30]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [31]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

/usr/local/lib/python3.11/dist-packages/langchain/chains/llm.py:370: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [32]:
new_examples[0]

{'qa_pairs': {'query': 'Based on the information provided, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) is Electronics.'}}

In [33]:
data[0]

Document(metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 0}, page_content='\ufeffid: AVqVGZNvQMlgsOJE6eUY\ndateAdded: 2017-03-03T16:56:05Z\ndateUpdated: 2018-10-25T16:36:31Z\nname: Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)\nasins: B00ZV9PXP2\nbrand: Amazon\ncategories: Computers,Electronics Features,Tablets,Electronics,iPad & Tablets,Kindle E-readers,iPad Accessories,Used:Tablets,E-Readers,E-Readers & Accessories,Computers/Tablets & Networking,Used:Computers Accessories,iPads Tablets,All Tablets,Tablets & E-readers,Computers & Tablets,Amazon,Tablets & eBook Readers\nprimaryCategories: Electronics\nimageURLs: https://pisces.bbystatic.com/image2/BestBuy_US/images/products/5442/5442403_sd.jpg,https://c1.neweggimages.com/NeweggImage/ProductImage/A3FA_1_201801081360871160.jpg,https://i.ebayimg.com/thumbs/images/g/N4IAAOSwoA9Zgkso/s-l96.jpg,http://i.ebayimg.com/thumbs/images/g/dpkAAOSwfpVZFKHy/s-l200.jpg,http://i.ebayimg.com/thumbs/images/g

In [34]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': 'Based on the information provided, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) is Electronics.'},
 {'query': 'What is the brand and model of the product described in the document?',
  'answer': 'The brand is Amazon and the model is Kindle E-Reader 6" Wifi (8th Generation, 2016).'},
 {'query': 'According to the document, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) is Electronics.'},
 {'query': 'What are some of the features that the reviewer, Kdperry, praises about the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) in their review?',
  'answer': 'Kdperry praises the portability of the Kindle, stating that it fits better in their purse than a giant book, and that they appreciate the fac

#### Combine examples

In [35]:
# examples += new_example
examples += d_flattened

In [36]:
examples[0]

{'query': 'Do customers like the battery life of this product?',
 'answer': 'Yes'}

In [37]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do customers like the battery life of this product?',
 'result': 'Based on the provided reviews, it seems that customers have mixed opinions about the battery life of this product. Some users mention that the battery life is not as long as advertised and recommend a backlit model, while others mention having to recharge the battery every 4-5 hours. So, it appears that there are differing views on the battery life of this product.'}

### Manual Evaluation - Fun part

In [38]:
import langchain
langchain.debug = True

In [39]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do customers like the battery life of this product?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do customers like the battery life of this product?",
  "context": "reviews.title: Battery life is awesome!\nreviews.username: simp141<<<<>>>>>reviews.text: The battery doesn't last as long as was advertised, if you have the money I would recommend the backlit model.\nreviews.title: Good reader\nreviews.username: Mark<<<<>>>>>reviews.sourceURLs: http://reviews.bestbuy.com/3545/5025500/reviews.htm?format=embedded&page=35,http://reviews.bestbuy.com/3545/5025500/reviews.htm?format=embedded&page=34,http://reviews.bestbuy.com/3545/5025500/reviews.htm?format=embedded&page=16,http://reviews.bestbuy.com/3545/5025500/reviews.htm?format=em

{'query': 'Do customers like the battery life of this product?',
 'result': 'Based on the provided reviews, it seems that customers have mixed opinions about the battery life of this product. Some users mention that the battery life is not as long as advertised and recommend a backlit model, while others mention having to recharge the battery every 4-5 hours. So, it appears that there are differing views on the battery life of this product.'}

In [40]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [41]:
examples += d_flattened

In [42]:
examples

[{'query': 'Do customers like the battery life of this product?',
  'answer': 'Yes'},
 {'query': 'Are users satisfied with the price?', 'answer': 'No'},
 {'query': 'Based on the information provided, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) is Electronics.'},
 {'query': 'What is the brand and model of the product described in the document?',
  'answer': 'The brand is Amazon and the model is Kindle E-Reader 6" Wifi (8th Generation, 2016).'},
 {'query': 'According to the document, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) is Electronics.'},
 {'query': 'What are some of the features that the reviewer, Kdperry, praises about the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016) in their review?',
 

In [43]:
predictions = qa.batch(examples)



> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [44]:
predictions

[{'query': 'Do customers like the battery life of this product?',
  'answer': 'Yes',
  'result': 'Based on the provided reviews, it seems that customers have mixed opinions about the battery life of this product. Some users mention that the battery life is not as long as advertised and recommend a backlit model, while others mention having to recharge the battery every 4-5 hours. So, it appears that there are differing views on the battery life of this product among customers.'},
 {'query': 'Are users satisfied with the price?',
  'answer': 'No',
  'result': 'Based on the provided reviews, users seem to be satisfied with the price of the tablet. One user mentioned that it is great if you have Prime and use Amazon services, while another user found it worth the price for serious readers and users of library e-books.'},
 {'query': 'Based on the information provided, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?',
  'answer': 'The primary categ

In [45]:
from langchain.evaluation.qa import QAEvalChain

In [46]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [47]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [48]:
graded_outputs

[{'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'}]

In [49]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Do customers like the battery life of this product?
Real Answer: Yes
Predicted Answer: Based on the provided reviews, it seems that customers have mixed opinions about the battery life of this product. Some users mention that the battery life is not as long as advertised and recommend a backlit model, while others mention having to recharge the battery every 4-5 hours. So, it appears that there are differing views on the battery life of this product among customers.

Example 1:
Question: Are users satisfied with the price?
Real Answer: No
Predicted Answer: Based on the provided reviews, users seem to be satisfied with the price of the tablet. One user mentioned that it is great if you have Prime and use Amazon services, while another user found it worth the price for serious readers and users of library e-books.

Example 2:
Question: Based on the information provided, what is the primary category of the Amazon Kindle E-Reader 6" Wifi (8th Generation, 2016)?
Real An

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [52]:
from langchain.document_loaders import CSVLoader

loader = CSVLoader(file_path="Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv")

index = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})
).from_loaders([loader])

llm = ChatOpenAI(temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)


In [53]:
# testing it out
question = "Are customers happy with the battery life of the Amazon Kindle?"
result = qa_chain.invoke({"query": question})
result["result"]

'Customers are generally happy with the battery life of the Amazon Kindle. In the reviews provided, users mention that the Kindle battery life is excellent, lasting for many hours or several days even with heavy usage. They appreciate the long battery life, especially when using the device daily for extended periods.'

In [54]:
result

{'query': 'Are customers happy with the battery life of the Amazon Kindle?',
 'result': 'Customers are generally happy with the battery life of the Amazon Kindle. In the reviews provided, users mention that the Kindle battery life is excellent, lasting for many hours or several days even with heavy usage. They appreciate the long battery life, especially when using the device daily for extended periods.',
 'source_documents': [Document(id='bc796571-5596-448b-815d-c3dd6288d0b8', metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 3346}, page_content="reviews.text: I or a family member have owned every version of the Kindle since it came out. This version has finally taken all of the best elements of the previous Kindles and combined them into one unit. It is small and light, has both touch screen and button functionality (depending on which you prefer - I'm a fan of the buttons), and an excellent back-light. Kindle battery life has always been excelle

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [55]:
eval_questions = [
    "Do customers like the battery life of this product?",
    "Is the sound quality considered good?",
    "Are buyers happy with the price?",
    "What do people think about the design?",
    "Is the product durable?"
]

eval_answers = [
    "Yes",
    "Yes",
    "No",
    "Most customers appreciate the design.",
    "It is considered durable by most users."
]
examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [56]:
examples

[{'query': 'Do customers like the battery life of this product?',
  'ground_truths': ['Yes']},
 {'query': 'Is the sound quality considered good?', 'ground_truths': ['Yes']},
 {'query': 'Are buyers happy with the price?', 'ground_truths': ['No']},
 {'query': 'What do people think about the design?',
  'ground_truths': ['Most customers appreciate the design.']},
 {'query': 'Is the product durable?',
  'ground_truths': ['It is considered durable by most users.']}]

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [57]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'Based on the provided reviews, the opinions on the sound quality of the device vary. Karrie and Anacortes both mentioned that the sound quality is good, with Karrie specifically stating that the quality of the sound is amazing. However, imakev expressed dissatisfaction with the sound quality, mentioning that it was pretty terrible with no bass and could be scratchy. So, the sound quality seems to be subjective and may vary depending on individual preferences.'

In [58]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [59]:
result_updated

{'question': 'Is the sound quality considered good?',
 'answer': 'Based on the provided reviews, the opinions on the sound quality of the device vary. Karrie and Anacortes both mentioned that the sound quality is good, with Karrie specifically stating that the quality of the sound is amazing. However, imakev expressed dissatisfaction with the sound quality, mentioning that it was pretty terrible with no bass and could be scratchy. So, the sound quality seems to be subjective and may vary depending on individual preferences.',
 'contexts': [Document(id='01ac5111-d9f7-41cf-8bd1-162cf2376b03', metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 1511}, page_content="reviews.text: Bought this for my parents' home and it's awesome. The quality of the sound is amazing. It's also helpful to get quick questions answered when they're running around or out of the house. I need to now purchase one for my own apartment. Also great when hosting guests.\nreviews.ti

In [60]:
!pip install --no-cache-dir recordclass

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 119.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for recordclass: filename=recordclass-0.23.1-cp311-cp311-linux_x86_64.whl size=453245 sha256=f741de0fc26d7c487f03298080099b9c2434ab05e79b5c960a6e694f413c7733
  Stored in directory: /tmp/pip-ephem-wheel-cache-7hlz_csz/wheels/23/43/c9/edc2de309801c14a6a7ecb6a38eb3844802c5e895ac3094a01
Successfully built recordclass


In [61]:
!pip install ragas==0.1.9

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 23.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.


In [62]:
from ragas.integrations.langchain import EvaluatorChain
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
)

# create evaluation chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_relevancy)
context_recall_chain = EvaluatorChain(metric=context_recall)

/usr/local/lib/python3.11/dist-packages/ragas/metrics/__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from ragas.metrics._answer_correctness import AnswerCorrectness, answer_correctness
/usr/local/lib/python3.11/dist-packages/ragas/metrics/__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to impo

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [63]:
# Recheck the result that we are going to validate.
result

{'query': 'Is the sound quality considered good?',
 'result': 'Based on the provided reviews, the opinions on the sound quality of the device vary. Karrie and Anacortes both mentioned that the sound quality is good, with Karrie specifically stating that the quality of the sound is amazing. However, imakev expressed dissatisfaction with the sound quality, mentioning that it was pretty terrible with no bass and could be scratchy. So, the sound quality seems to be subjective and may vary depending on individual preferences.',
 'source_documents': [Document(id='01ac5111-d9f7-41cf-8bd1-162cf2376b03', metadata={'source': 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'row': 1511}, page_content="reviews.text: Bought this for my parents' home and it's awesome. The quality of the sound is amazing. It's also helpful to get quick questions answered when they're running around or out of the house. I need to now purchase one for my own apartment. Also great when hosting guests.\nrevie

**Faithfulness**

In [75]:
result = qa_chain.invoke({"query": "Do customers like the battery life?"})

prepared_input = {
    "question": "Do customers like the battery life?",
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]]
}

eval_result = faithfulness_chain(prepared_input)

print("✅ Raw eval result:", eval_result)


✅ Raw eval result: {'question': 'Do customers like the battery life?', 'answer': "Customers have mixed opinions about the battery life of the product. Some customers find the battery life to be short and needing frequent recharges, while others mention that the battery life isn't great but still consider the product good for the price. One customer even recommends a backlit model if battery life is a concern.", 'contexts': ["reviews.text: The battery doesn't last as long as was advertised, if you have the money I would recommend the backlit model.\nreviews.title: Good reader\nreviews.username: Mark", 'reviews.title: Battery life is awesome!\nreviews.username: simp141', "reviews.text: Battery life isn't great, but overall for what I paid it is a good tablet for my kids. I did inadvertently purchase the wrong color and when I returned to the store to exchange it, the store mgr was a bit of a jerk and would not allow me to do so, despite being Elite Plus and being under 45 days. That asid

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [78]:
result = qa_chain.invoke({"query": "Do customers like the battery life?"})

prepared_input = {
    "question": "Do customers like the battery life?",
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]]
}

fake_input = prepared_input.copy()
fake_input["answer"] = "we are the champions"

fake_eval_result = faithfulness_chain(fake_input)

print("Fake Faithfulness Score:", fake_eval_result.get("faithfulness_score", fake_eval_result.get("score")))

Fake Faithfulness Score: None


**Context Relevancy**

In [82]:
result = qa_chain.invoke({"query": "Do customers like the battery life?"})

context_recall_input = {
    "question": "Do customers like the battery life?",
    "ground_truth": "Yes",
    "contexts": [doc.page_content for doc in result["source_documents"]]
}

eval_result = context_recall_chain(context_recall_input)

print("Context Recall Score:", eval_result.get("context_recall_score", eval_result.get("score")))

Context Recall Score: None


High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [84]:
from langchain.schema import Document

fake_context_input = {
    "question": "Do customers like the battery life?",
    "ground_truth": "Yes",
    "contexts": ["I love christmas"]
}

eval_result = context_recall_chain(fake_context_input)

print("Fake Context Recall Score:", eval_result.get("context_recall_score", eval_result.get("score")))


Fake Context Recall Score: None


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [92]:
!pip install datasets --quiet

from datasets import Dataset
from ragas import evaluate

ragas_dataset = Dataset.from_list(ragas_ready_examples)

print("evaluating...")
r = evaluate(
    ragas_dataset,
    metrics=[faithfulness]
)

r

evaluating...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.8900}

In [94]:
from ragas import evaluate
from datasets import Dataset

ragas_dataset = Dataset.from_list(ragas_ready_examples)

print("evaluating context recall...")
r = evaluate(
    ragas_dataset,
    metrics=[context_recall]
)

r


evaluating context recall...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.0000}

In [95]:
print(examples[0])


{'query': 'Do customers like the battery life of this product?', 'ground_truths': ['Yes']}
